# servo_bus: Feetech STS3215 packets and the bus driver

Library notebook for `gambit_arm`. The SO-ARM101 uses six Feetech STS3215 serial bus servos
on one half-duplex TTL line, driven through the Waveshare Bus Servo Adapter (USB serial).
This notebook builds and parses SCS/STS protocol packets (pure bytes, fully tested here) and
sketches a `ServoBus` class on top of pyserial (I/O left as TODO, needs the hardware).

The same protocol is implemented in C++ (`gambit_cpp/libs/servo_bus/sts_protocol.cpp`) and
C (`firmware/stm32/Core/Src/servo_bus.c`); all three must produce identical bytes. The
self-check vectors below are also in `gambit_cpp/tests/test_sts_protocol.cpp`.

Milestone: *Hardware and execution* (servo IDs assigned, calibrated, teleoperating).

**TODO**
- [x] packet encode (PING / READ / WRITE / SYNC_WRITE) and checksum
- [x] status packet decode with checksum check
- [x] ticks <-> radians (4096 ticks per revolution)
- [ ] verify every control-table address against the STS3215 datasheet (marked VERIFY)
- [ ] `ServoBus` I/O: open port, transact with timeout, handle half-duplex echo
- [ ] per-joint calibration (offset and direction) from run_servo_setup.ipynb
- [ ] SYNC_READ of present positions (one round trip for all six servos)
- [ ] measure the round-trip time at 1 Mbaud (needed for the Python vs C++ comparison)

In [ ]:
import math
import time
from dataclasses import dataclass, field

## Packet format

```
instruction:  0xFF 0xFF  ID  LEN  INSTR  P1 .. PN  CHK      LEN = N + 2
status:       0xFF 0xFF  ID  LEN  ERROR  P1 .. PN  CHK
CHK = ~(ID + LEN + INSTR/ERROR + P1 + ... + PN) & 0xFF
```

ID 0xFE is broadcast (no reply). STS servos store 16-bit registers **little-endian**
(low byte first); the older SCS series is big-endian, so do not copy SCS examples blindly.

In [ ]:
HEADER = bytes([0xFF, 0xFF])
BROADCAST_ID = 0xFE

class Inst:
    PING = 0x01
    READ = 0x02
    WRITE = 0x03
    REG_WRITE = 0x04
    ACTION = 0x05
    SYNC_READ = 0x82
    SYNC_WRITE = 0x83

# Control table (EEPROM below 0x28, RAM from 0x28). VERIFY against the STS3215 datasheet /
# Feetech memory table before writing anything, especially EEPROM (ID, baud, limits, offset).
class Reg:
    ID = 0x05                   # VERIFY
    BAUD_RATE = 0x06            # VERIFY (0 = 1 Mbaud)
    MIN_ANGLE_LIMIT = 0x09      # 2 bytes, VERIFY
    MAX_ANGLE_LIMIT = 0x0B      # 2 bytes, VERIFY
    POSITION_OFFSET = 0x1F      # 2 bytes, VERIFY
    TORQUE_ENABLE = 0x28        # VERIFY
    ACCELERATION = 0x29         # VERIFY
    GOAL_POSITION = 0x2A        # 2 bytes, VERIFY
    GOAL_TIME = 0x2C            # 2 bytes, VERIFY
    GOAL_SPEED = 0x2E           # 2 bytes, VERIFY
    TORQUE_LIMIT = 0x30         # 2 bytes, 0..1000, VERIFY
    LOCK = 0x37                 # EEPROM write lock, VERIFY
    PRESENT_POSITION = 0x38     # 2 bytes, VERIFY
    PRESENT_SPEED = 0x3A        # 2 bytes, VERIFY
    PRESENT_LOAD = 0x3C         # 2 bytes, VERIFY
    PRESENT_VOLTAGE = 0x3E      # VERIFY
    PRESENT_TEMPERATURE = 0x3F  # VERIFY
    MOVING = 0x42               # VERIFY
    PRESENT_CURRENT = 0x45      # 2 bytes, VERIFY

# Error bits in a status packet (VERIFY).
ERROR_BITS = {0: "voltage", 1: "angle", 2: "overheat", 3: "overele", 5: "overload"}

def checksum(body):
    """body = ID, LEN, INSTR/ERROR, params (everything after the 0xFF 0xFF header)."""
    return (~sum(body)) & 0xFF

def encode(servo_id, instruction, params=b""):
    params = bytes(params)
    if len(params) > 250:
        raise ValueError("too many params")
    body = bytes([servo_id, len(params) + 2, instruction]) + params
    return HEADER + body + bytes([checksum(body)])

def u16_le(v):
    return bytes([v & 0xFF, (v >> 8) & 0xFF])

def from_u16_le(lo, hi):
    return lo | (hi << 8)

def ping(servo_id):
    return encode(servo_id, Inst.PING)

def read(servo_id, address, length):
    return encode(servo_id, Inst.READ, [address, length])

def write(servo_id, address, data):
    return encode(servo_id, Inst.WRITE, bytes([address]) + bytes(data))

def sync_write(address, data_length, entries):
    """entries: {servo_id: data bytes (len == data_length)} -> one broadcast packet."""
    params = bytearray([address, data_length])
    for sid, data in entries.items():
        if len(data) != data_length:
            raise ValueError("sync_write: wrong data length")
        params += bytes([sid]) + bytes(data)
    return encode(BROADCAST_ID, Inst.SYNC_WRITE, params)

@dataclass
class Status:
    id: int
    error: int
    params: bytes

    @property
    def errors(self):
        return [name for bit, name in ERROR_BITS.items() if self.error & (1 << bit)]

def decode_status(data):
    """Parse the first status packet in `data` (garbage before the header is skipped).
    Returns (Status, bytes_consumed) or (None, 0) if incomplete / bad checksum."""
    data = bytes(data)
    i = data.find(HEADER)
    if i < 0 or len(data) < i + 6:
        return None, 0
    length = data[i + 3]
    end = i + 4 + length                 # index one past the checksum
    if length < 2 or len(data) < end:
        return None, 0
    body = data[i + 2 : end - 1]
    if checksum(body) != data[end - 1]:
        return None, 0
    return Status(id=data[i + 2], error=data[i + 4], params=data[i + 5 : end - 1]), end

## Position units

4096 ticks per revolution. We map tick 2048 to 0 rad (servo centered); the per-joint
calibration (offset and sign so that 0 rad matches the URDF zero) is a TODO filled by
run_servo_setup.ipynb and stored in `CALIBRATION` below.

In [ ]:
TICKS_PER_REV = 4096
CENTER_TICK = 2048

def ticks_to_rad(ticks):
    return (ticks - CENTER_TICK) * 2 * math.pi / TICKS_PER_REV

def rad_to_ticks(rad):
    return int(min(TICKS_PER_REV - 1, max(0, CENTER_TICK + round(rad * TICKS_PER_REV / (2 * math.pi)))))

# LeRobot SO-101 convention: IDs 1..6 in this order (verify after assigning IDs).
JOINT_IDS = {"shoulder_pan": 1, "shoulder_lift": 2, "elbow_flex": 3,
             "wrist_flex": 4, "wrist_roll": 5, "gripper": 6}

@dataclass
class JointCalibration:
    offset_ticks: int = 0      # added to the raw reading so URDF zero reads CENTER_TICK
    sign: int = 1              # +1 or -1 if the servo turns opposite to the URDF axis

# TODO: fill from run_servo_setup.ipynb (save/load as YAML next to board.yaml, e.g.
# gambit_bringup/config/arm_calibration.yaml).
CALIBRATION = {name: JointCalibration() for name in JOINT_IDS}

def joint_rad_from_raw(name, raw_ticks, cal=CALIBRATION):
    c = cal[name]
    return c.sign * ticks_to_rad(raw_ticks + c.offset_ticks)

def raw_from_joint_rad(name, rad, cal=CALIBRATION):
    c = cal[name]
    return rad_to_ticks(c.sign * rad) - c.offset_ticks

## ServoBus (pyserial)

Thin wrapper: one method per instruction. pyserial is imported lazily so this notebook runs
without it. The adapter is half-duplex: depending on the board you may read back your own
transmitted bytes before the reply ("echo"); `transact` must skip them.

In [ ]:
@dataclass
class ServoBus:
    port: str = "/dev/ttyACM0"
    baud: int = 1_000_000
    timeout_s: float = 0.02
    ser: object = field(default=None, repr=False)

    def open(self):
        import serial  # pyserial
        self.ser = serial.Serial(self.port, self.baud, timeout=self.timeout_s)
        return self

    def close(self):
        if self.ser is not None:
            self.ser.close()
            self.ser = None

    def __enter__(self):
        return self.open()

    def __exit__(self, *exc):
        self.close()

    def transact(self, packet, expect_reply=True):
        """Send one instruction packet and return the decoded Status (or None).
        TODO: implement: reset_input_buffer(); write(packet); flush(); if the adapter echoes,
        read and discard len(packet) bytes; read the 6-byte minimum status, then the rest from
        its LEN byte; decode_status(); retry once on timeout/bad checksum; raise on error bits."""
        raise NotImplementedError("ServoBus.transact needs the real adapter")

    def ping(self, servo_id):
        return self.transact(ping(servo_id)) is not None

    def read_u16(self, servo_id, address):
        st = self.transact(read(servo_id, address, 2))
        return from_u16_le(st.params[0], st.params[1])

    def write_u8(self, servo_id, address, value):
        return self.transact(write(servo_id, address, [value]), expect_reply=servo_id != BROADCAST_ID)

    def write_u16(self, servo_id, address, value):
        return self.transact(write(servo_id, address, u16_le(value)), expect_reply=servo_id != BROADCAST_ID)

    def torque(self, servo_id, enable):
        return self.write_u8(servo_id, Reg.TORQUE_ENABLE, 1 if enable else 0)

    def set_goal_ticks(self, goals):
        """goals: {servo_id: ticks}. One SYNC_WRITE, no reply."""
        return self.transact(sync_write(Reg.GOAL_POSITION, 2, {i: u16_le(t) for i, t in goals.items()}),
                             expect_reply=False)

    def read_positions(self, ids=tuple(JOINT_IDS.values())):
        """TODO: replace the per-servo loop with one SYNC_READ round trip."""
        return {i: self.read_u16(i, Reg.PRESENT_POSITION) for i in ids}

    def set_joint_targets(self, q_rad):
        """q_rad: {joint name: rad}. Callers must pass targets through safety.validate_command first."""
        return self.set_goal_ticks({JOINT_IDS[n]: raw_from_joint_rad(n, v) for n, v in q_rad.items()})

    def round_trip_time(self, servo_id=1, n=100):
        """Mean seconds per PING round trip (Python path number for the C++ comparison)."""
        t0 = time.perf_counter()
        for _ in range(n):
            self.ping(servo_id)
        return (time.perf_counter() - t0) / n

## Self-check

In [ ]:
assert ping(1) == bytes([0xFF, 0xFF, 0x01, 0x02, 0x01, 0xFB])          # ~(1+2+1) = 0xFB
assert checksum([0xFE, 0xFF, 0x10]) == (~(0xFE + 0xFF + 0x10)) & 0xFF
p = read(3, Reg.PRESENT_POSITION, 2)
assert len(p) == 8 and p[3] == 4 and p[-1] == (~(3 + 4 + 2 + 0x38 + 2)) & 0xFF
assert write(1, Reg.GOAL_POSITION, u16_le(2048)) == bytes([0xFF, 0xFF, 1, 5, 3, 0x2A, 0x00, 0x08,
                                                           (~(1 + 5 + 3 + 0x2A + 0 + 8)) & 0xFF])
sw = sync_write(Reg.GOAL_POSITION, 2, {1: u16_le(1000), 2: u16_le(3000)})
assert sw[2] == BROADCAST_ID and sw[3] == 2 + 6 + 2 and len(sw) == 14
# status round trip with leading garbage
body = bytes([1, 4, 0, 0x00, 0x08])
reply = b"\x12\x34" + HEADER + body + bytes([checksum(body)])
st, n = decode_status(reply)
assert st.id == 1 and st.error == 0 and from_u16_le(*st.params) == 2048 and n == len(reply)
assert decode_status(HEADER + bytes([1, 2, 0, 0]))[0] is None          # bad checksum
assert decode_status(HEADER + bytes([1]))[0] is None                   # incomplete
assert Status(1, 0b100100, b"").errors == ["overheat", "overload"]
assert ticks_to_rad(2048) == 0 and abs(ticks_to_rad(3072) - math.pi / 2) < 1e-12
assert all(rad_to_ticks(ticks_to_rad(t)) == t for t in (0, 100, 2048, 4000, 4095))
assert rad_to_ticks(100.0) == 4095
assert raw_from_joint_rad("elbow_flex", joint_rad_from_raw("elbow_flex", 1500)) == 1500
print("servo_bus OK")